In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report, precision_recall_curve
from catboost import CatBoostClassifier

RANDOM_STATE = 42
TARGET = "delayed"

df_cb = pd.read_csv("model_ready_catboost.csv")

X = df_cb.drop(columns=[TARGET])
y = df_cb[TARGET].astype(int)

# final untouched test set
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

# validation set for threshold selection
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2,
    stratify=y_train_full,
    random_state=RANDOM_STATE
)

cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
cat_features = [X_train.columns.get_loc(c) for c in cat_cols]

print(X_train.shape, X_val.shape, X_test.shape)
print("Categorical columns:", cat_cols)

(136837, 18) (34210, 18) (42762, 18)
Categorical columns: ['Courier Partner', 'Pickup State', 'Drop State', 'Payment Mode', 'Zone', 'state_lane', 'courier_zone', 'order_dow']


In [2]:
def best_threshold_f1(y_true, y_prob):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-9)
    best_idx = np.nanargmax(f1[:-1])   # thresholds is shorter by 1
    thr = float(thresholds[best_idx])
    return thr, float(f1[best_idx]), float(precision[best_idx]), float(recall[best_idx])

def eval_model(name, y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    print(f"\n=== {name} ===")
    print("Threshold:", thr)
    print("ROC-AUC:", roc_auc_score(y_true, y_prob))
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred))

In [3]:
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
class_weights = [1.0, neg / max(pos, 1)]

param_grid = [
    {"depth": d, "learning_rate": lr, "l2_leaf_reg": l2}
    for d in [6, 8]
    for lr in [0.03, 0.05]
    for l2 in [3, 10]
]

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

best_auc = -1
best_params = None
grid_results = []

for params in param_grid:
    fold_aucs = []
    print(f"\nRunning params: {params}")

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train), start=1):
        X_tr = X_train.iloc[tr_idx]
        X_va = X_train.iloc[va_idx]
        y_tr = y_train.iloc[tr_idx]
        y_va = y_train.iloc[va_idx]

        cb = CatBoostClassifier(
            iterations=2000,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=RANDOM_STATE,
            verbose=False,
            class_weights=class_weights,
            task_type="GPU",
            devices="0",
            od_type="Iter",
            od_wait=100,
            **params
        )

        cb.fit(
            X_tr, y_tr,
            cat_features=cat_features,
            eval_set=(X_va, y_va),
            use_best_model=True
        )

        y_prob_va = cb.predict_proba(X_va)[:, 1]
        auc = roc_auc_score(y_va, y_prob_va)
        fold_aucs.append(auc)
        print(f"  Fold {fold}: AUC = {auc:.5f}")

    mean_auc = float(np.mean(fold_aucs))
    std_auc = float(np.std(fold_aucs))

    grid_results.append({
        "params": params,
        "mean_auc": mean_auc,
        "std_auc": std_auc
    })

    print(f"Mean CV AUC: {mean_auc:.5f} +/- {std_auc:.5f}")

    if mean_auc > best_auc:
        best_auc = mean_auc
        best_params = params

print("\nBest Params:", best_params)
print("Best CV AUC:", best_auc)


Running params: {'depth': 6, 'learning_rate': 0.03, 'l2_leaf_reg': 3}


Default metric period is 5 because AUC is/are not implemented for GPU


  Fold 1: AUC = 0.80390


Default metric period is 5 because AUC is/are not implemented for GPU


  Fold 2: AUC = 0.80667


...

  Fold 1: AUC = 0.80369


Default metric period is 5 because AUC is/are not implemented for GPU


  Fold 2: AUC = 0.80715


Default metric period is 5 because AUC is/are not implemented for GPU


  Fold 3: AUC = 0.80660
Mean CV AUC: 0.80581 +/- 0.00152

Best Params: {'depth': 6, 'learning_rate': 0.05, 'l2_leaf_reg': 10}
Best CV AUC: 0.8062710667777496


In [4]:
best_cb = CatBoostClassifier(
    iterations=3000,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RANDOM_STATE,
    verbose=200,
    class_weights=class_weights,
    task_type="GPU",
    devices="0",
    od_type="Iter",
    od_wait=150,
    **best_params
)

best_cb.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_val, y_val),
    use_best_model=True
)

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7449460	best: 0.7449460 (0)	total: 30ms	remaining: 1m 30s
200:	test: 0.8001694	best: 0.8001694 (200)	total: 6.27s	remaining: 1m 27s
400:	test: 0.8054775	best: 0.8054775 (400)	total: 12.4s	remaining: 1m 20s
600:	test: 0.8079292	best: 0.8079292 (600)	total: 18.2s	remaining: 1m 12s
800:	test: 0.8091098	best: 0.8091142 (799)	total: 24.4s	remaining: 1m 6s
1000:	test: 0.8102765	best: 0.8102798 (999)	total: 30.5s	remaining: 1m
1200:	test: 0.8109304	best: 0.8109304 (1200)	total: 36.5s	remaining: 54.7s
1400:	test: 0.8113244	best: 0.8113359 (1398)	total: 42.9s	remaining: 48.9s
1600:	test: 0.8118198	best: 0.8118198 (1600)	total: 49s	remaining: 42.8s
1800:	test: 0.8121689	best: 0.8121689 (1800)	total: 55.1s	remaining: 36.7s
2000:	test: 0.8124539	best: 0.8124539 (2000)	total: 1m 1s	remaining: 30.7s
2200:	test: 0.8127381	best: 0.8127405 (2192)	total: 1m 7s	remaining: 24.5s
2400:	test: 0.8128751	best: 0.8128999 (2385)	total: 1m 14s	remaining: 18.5s
2600:	test: 0.8129883	best: 0.8130112 (25

CatBoostClassifier(class_weights=[1.0, np.float64(3.944426377597109)], depth=6, devices='0', eval_metric='AUC', iterations=3000, l2_leaf_reg=10, learning_rate=0.05, loss_function='Logloss', od_type='Iter', od_wait=150, random_seed=42, task_type='GPU', verbose=200)

In [5]:
cb_val_prob = best_cb.predict_proba(X_val)[:, 1]

cb_thr, cb_f1, cb_p, cb_r = best_threshold_f1(y_val, cb_val_prob)

print("Best threshold from validation:", cb_thr)
print("Validation F1:", cb_f1)
print("Validation Precision:", cb_p)
print("Validation Recall:", cb_r)

Best threshold from validation: 0.5810815089845666
Validation F1: 0.5435967297554635
Validation Precision: 0.4755661501787843
Validation Recall: 0.6343402225755167


In [6]:
cb_test_prob = best_cb.predict_proba(X_test)[:, 1]

eval_model("CatBoost GPU Grid Search", y_test, cb_test_prob, cb_thr)


=== CatBoost GPU Grid Search ===
Threshold: 0.5810815089845666
ROC-AUC: 0.8084997830233146
Confusion Matrix:
 [[27908  6206]
 [ 3259  5389]]
              precision    recall  f1-score   support

           0       0.90      0.82      0.86     34114
           1       0.46      0.62      0.53      8648

    accuracy                           0.78     42762
   macro avg       0.68      0.72      0.69     42762
weighted avg       0.81      0.78      0.79     42762



In [7]:
grid_df = pd.DataFrame(grid_results)
grid_df = grid_df.sort_values("mean_auc", ascending=False)
print(grid_df)

                                              params  mean_auc   std_auc
3  {'depth': 6, 'learning_rate': 0.05, 'l2_leaf_r...  0.806271  0.001287
2  {'depth': 6, 'learning_rate': 0.05, 'l2_leaf_r...  0.806212  0.001325
5  {'depth': 8, 'learning_rate': 0.03, 'l2_leaf_r...  0.806201  0.001453
1  {'depth': 6, 'learning_rate': 0.03, 'l2_leaf_r...  0.805872  0.001319
0  {'depth': 6, 'learning_rate': 0.03, 'l2_leaf_r...  0.805853  0.001389
7  {'depth': 8, 'learning_rate': 0.05, 'l2_leaf_r...  0.805812  0.001520
4  {'depth': 8, 'learning_rate': 0.03, 'l2_leaf_r...  0.805729  0.001642
6  {'depth': 8, 'learning_rate': 0.05, 'l2_leaf_r...  0.805074  0.001606


In [8]:
feat_imp = pd.DataFrame({
    "feature": X_train.columns,
    "importance": best_cb.get_feature_importance()
}).sort_values("importance", ascending=False)

print(feat_imp.head(15))

                    feature  importance
14               state_lane   11.798766
15             courier_zone   10.089589
1           shipment_volume   10.087096
10             Pickup State    9.654236
16                order_dow    8.692578
3                   Max Tat    8.662922
11               Drop State    7.468868
4                   density    7.285240
9           Courier Partner    6.783902
0           Shipment Weight    4.750040
5   chargeable_weight_proxy    4.662612
13                     Zone    2.756723
2            Total Quantity    2.361235
7            promise_window    2.347727
6                 cod_ratio    2.205103
